# Extraction — image → identities

Produces `<RESULTS_ROOT>/<RUN_NAME>/extraction/`. No analysis-tunable
params; anything that depends on plot styling or geometry lives in
`notebooks/analysis.ipynb`.

In [ ]:
# Papermill-injected defaults (overridden by run_experiment.py).
RUN_NAME = "control_experiment"
RESULTS_ROOT = "../results"
STACK_PATH = "../data/Control-experiment/Grad LB+sucr-20-0.zvi  Ch0.tif"
FLUOR_PATH = "../data/Control-experiment/Grad LB+sucr-20-0.zvi  Ch1-BG.tif"
MODEL_TYPE = "cyto3"
DETECT_PARAMS = {
    "diameter": 32, "min_area": 300, "min_circularity": 0.7,
    "min_contrast": 1250, "exclude_edges": True,
    "gpu": True, "resample": False,
}
GATING_Z_THRESHOLD = 3.5
SEARCH_RANGE = 30.0
MEMORY = 3
MERGE_MAX_DISTANCE = 15.0
MERGE_MAX_GAP = 18
MIN_TRACK_DETECTIONS = 4
NUCLEUS_DIAMETER = 25
NUCLEUS_MIN_AREA = 100

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, "../src")

from cell_analysis import (
    load_experiment, detect_cells_stack, run_frame_gating,
    run_tracking, filter_short_tracks, detect_nuclei_stack,
    plot_frame_preview, plot_detections, plot_channels_preview,
    plot_frame_gating,
    EXTRACTION_PARAM_NAMES, finalize_extraction_run,
)

EXTRACTION_DIR = Path(RESULTS_ROOT) / RUN_NAME / "extraction"
EXTRACTION_DIR.mkdir(parents=True, exist_ok=True)
print(f"Extraction outputs will be written to {EXTRACTION_DIR}")

## 1. Load stacks

In [ ]:
phase_stack, fluor_stack = load_experiment(STACK_PATH, FLUOR_PATH)
plot_frame_preview(phase_stack)
plot_channels_preview(phase_stack, fluor_stack)

## 2. Cellpose detection (phase channel)

In [ ]:
centroids_all, label_stack = detect_cells_stack(
    phase_stack, model_type=MODEL_TYPE, **DETECT_PARAMS,
)
plot_detections(phase_stack[0], centroids_all[0])

## 3. Frame gating

In [ ]:
detections, bad_frames, diagnostics = run_frame_gating(
    label_stack, z_threshold=GATING_Z_THRESHOLD, results_dir=EXTRACTION_DIR,
)
plot_frame_gating(diagnostics, bad_frames)

## 4. Tracking + merging + minimum lifetime

In [ ]:
tracked, track_stats, merge_log = run_tracking(
    detections,
    search_range=SEARCH_RANGE, memory=MEMORY,
    merge_max_distance=MERGE_MAX_DISTANCE, merge_max_gap=MERGE_MAX_GAP,
)
tracked, track_stats = filter_short_tracks(
    tracked, track_stats, min_detections=MIN_TRACK_DETECTIONS,
)

## 5. Nucleus detection (fluor channel)

In [ ]:
nucleus_label_stack = detect_nuclei_stack(
    fluor_stack, diameter=NUCLEUS_DIAMETER, min_area=NUCLEUS_MIN_AREA,
    gpu=DETECT_PARAMS.get("gpu", False),
)

## 6. Persist extraction bundle

In [ ]:
# Persist the extraction bundle.
# `finalize_extraction_run` builds the provenance dict, derives
# dropped_frames from `diagnostics`, and writes all 8 artifacts under
# EXTRACTION_DIR. The `params` dict is built from the canonical
# EXTRACTION_PARAM_NAMES list so adding a new param means one edit in
# src/cell_analysis/io.py — the notebook doesn't need updating here.
params = {k: globals()[k] for k in EXTRACTION_PARAM_NAMES}

provenance = finalize_extraction_run(
    EXTRACTION_DIR,
    label_stack=label_stack,
    nucleus_label_stack=nucleus_label_stack,
    tracked=tracked, track_stats=track_stats,
    diagnostics=diagnostics, merge_log=merge_log,
    params=params,
    stack_path=STACK_PATH, fluor_path=FLUOR_PATH,
    repo_root=Path("..").resolve(),
)
print(f"Wrote extraction bundle to {EXTRACTION_DIR}")